# CUTEst

In [ ]:
import json
import os

from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.problem.cutest_noised import CUTEstNoisedProblem
from qnlab.solver.qn import qn
from qnlab.util.callback import Callback, CallbackTimeoutError
from qnlab.util.method import Method, get_methods
from qnlab.experiment.for_cutest_run import run, load_results
from qnlab.experiment.for_cutest_vis import draw_pp, individual_plot

## Original paper experiments

The following cell reproduces the original CUTEst runs and plots. A full run is long-running.

In [ ]:
working_directory = Path.cwd().resolve()
if (working_directory / "pyproject.toml").exists():
    repository_root = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_root = working_directory.parent
else:
    raise RuntimeError("Open this notebook from the repository root or notebooks/.")
os.chdir(repository_root)
print(repository_root)

In [ ]:
ERROR_CAUSING_TASKS = [
    (16, "INDEFM", "SciPy"),
    (16, "INDEFM", "NTRQN"),
    (16, "INDEFM", "NTRQN-MS"),
    (16, "INDEFM", "Reg-Sec"),
    (16, "OSCIGRAD", "NTRQN-MS"),
    (32, "INDEFM", "NTRQN"),
    (32, "INDEFM", "NTRQN-MS"),
    (32, "OSCIGRAD", "NTRQN-MS"),
]

for precision, noise in [
    (64, np.float64(0.0)),
    (32, np.float64(0.0)),
    (16, np.float64(0.0)),
    (64, np.float64(1e-3)),
]:
    problems = problemsToRun(precision)[1:]
    methods, *_ = get_methods()
    if noise > 0:
        new_methods = []
        for method, option in methods:
            new_option = option.copy()
            new_option["gtol"] = noise * 10
            new_methods.append((method, new_option))
        methods = new_methods
    run(problems, methods, precision, noise, ERROR_CAUSING_TASKS, TL=600)

methods, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
for precision, _noise, _gtol in [
    (64, 0.0, 1e-1),
    (64, 0.0, 1e-3),
    (64, 0.0, 1e-5),
    (32, 0.0, 1e-1),
    (32, 0.0, 1e-3),
    (32, 0.0, 1e-5),
    (16, 0.0, 1e-1),
    (16, 0.0, 1e-3),
    (16, 0.0, 1e-5),
    (64, 1e-3, 1e-2),
]:
    noise = np.float64(_noise)
    gtol = np.float64(_gtol)
    problems = problemsToRun(precision)

    alg_names, callsM, fxsM, gnormsM, problems = load_results(
        methods, problems, precision, noise, gtol
    )
    draw_pp(
        alg_names,
        callsM,
        ALGORITHM_COLORS,
        ALGORITHM_LINE_STYLES,
        precision,
        noise,
        gtol,
    )

    if precision == 64 and gtol == 1e-5:
        individual_plot(problems, methods, precision, noise)
    if True:
        data = {}
        data["problem"] = problems
        for i, alg_name in enumerate(alg_names):
            data[f"{alg_name}"] = callsM[i, :].tolist()
        df = pd.DataFrame(data)
        df.set_index("problem", inplace=True)

        # Apply styling
        def color_scale_with_cmap(row):
            if np.all(np.isinf(row.values)):
                return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
            norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)  # type:ignore
            cmap = matplotlib.colormaps["coolwarm"]
            return [
                f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
                for r, g, b, _ in cmap(norm(row.values))
            ]

        styled_df = df.style.apply(color_scale_with_cmap, axis=1)
        display(styled_df)


## Additional experiments for the MPC response

These experiments separate function and gradient noise, vary the assumed function-error bound, and add the OFFO, restart, and default-termination NTQN variants requested for the MPC response. Results are stored under `data/temp/` and overwrite earlier results for the same condition.

First select a small set of problems and leave `RUN_EXPERIMENTS = False` to inspect the task list. Set it to `True` only when ready. Running all problems is long-running.

In [ ]:
NOISE_SCENARIOS = {
    # Theory-aligned experiment: the gradient oracle is exact.
    "function_only": (1e-3, 0.0, 1e-2),
    # Deliberately adverse stress test outside the exact-gradient theorem.
    "joint_noise": (1e-3, 1e-3, 1e-2),
    # Sensitivity to the supplied function-error estimate.
    "eps_under": (1e-3, 0.0, 1e-4),
    "eps_nominal": (1e-3, 0.0, 1e-2),
    "eps_over": (1e-3, 0.0, 1e-1),
}

STANDARD_LABELS = {
    "NTRQN", "NTRQN-MS", "Line", "Line-MS",
    "Reg", "Reg-Sec", "SciPy", "NTQN",
}
SCENARIO_DEFAULT_LABELS = {
    "function_only": STANDARD_LABELS
    | {"NTRQN-OFFO", "NTRQN-Restart", "NTQN-Default-Termination"},
    "joint_noise": STANDARD_LABELS | {"NTQN-Default-Termination"},
    "eps_under": {"NTRQN", "NTRQN-MS"},
    "eps_nominal": {"NTRQN", "NTRQN-MS"},
    "eps_over": {"NTRQN", "NTRQN-MS"},
}

SCENARIOS_TO_RUN = list(NOISE_SCENARIOS)
SEEDS = [0]
PROBLEMS_TO_RUN = None  # For a quick check, use e.g. ["ROSENBR"].
METHODS_TO_RUN = None  # For a quick check, use e.g. ["NTRQN"].
TIME_LIMIT = 600.0
MAX_ITERATIONS = 15_000
RUN_EXPERIMENTS = False

In [ ]:
RESULT_ROOT = Path("data/temp")


def get_mpc_response_methods(max_iterations):
    methods, _, _ = get_methods(m=10, MI=max_iterations)
    methods.extend(
        [
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-OFFO"),
                {"m": 10, "max_iterations": max_iterations, "force_offo": 1},
            ),
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-Restart"),
                {"m": 10, "max_iterations": max_iterations, "restart_threshold": 1.0},
            ),
            (
                Method("NTQN", "raw", "raw", "bfgs", label="NTQN-Default-Termination"),
                {"m": 10, "max_iterations": max_iterations, "terminate": 3, "stop_at_gtol": 0},
            ),
        ]
    )
    return methods


def get_problem_names(selected=None):
    with Path("data/CUTEst/valid_problems.json").open(encoding="utf-8") as file:
        available = json.load(file)["valid_problems"]["precision_64"]
    names = [entry["name"] if isinstance(entry, dict) else entry for entry in available]
    if selected is None:
        return names
    unknown = sorted(set(selected) - set(names))
    if unknown:
        raise ValueError(f"Problems not in the 64-bit valid list: {unknown}")
    return selected


def result_path(scenario, seed, problem, label):
    return RESULT_ROOT / scenario / f"seed_{seed}" / problem / f"{label}.npz"


def run_task(scenario, seed, problem_name, method, options, time_limit):
    function_noise, gradient_noise, assumed_error = NOISE_SCENARIOS[scenario]
    path = result_path(scenario, seed, problem_name, method.label)

    problem = CUTEstNoisedProblem(
        problem_name,
        precision=64,
        function_noise=np.float64(function_noise),
        gradient_noise=np.float64(gradient_noise),
        assumed_function_error=np.float64(assumed_error),
        seed=seed,
    )
    callback = Callback(time_limit=time_limit)
    status = "completed"
    error = ""
    try:
        qn(problem, method, options, callback)
    except CallbackTimeoutError as exc:
        status = "timeout"
        error = str(exc)
    except Exception as exc:
        status = "error"
        error = repr(exc)

    path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        "scenario": scenario,
        "seed": seed,
        "problem": problem_name,
        "dimension": int(problem.n),
        "method": method.label,
        "options": options,
        "function_noise": function_noise,
        "gradient_noise": gradient_noise,
        "assumed_function_error": assumed_error,
        "status": status,
        "error": error,
        "diagnostics": dict(callback.others),
    }
    np.savez_compressed(
        path,
        calls=np.asarray(callback.calls),
        fxs=np.asarray(callback.fxs),
        gnorms=np.asarray(callback.gnorms),
        times=np.asarray(callback.times),
        metadata=json.dumps(metadata),
    )
    print(f"SAVE {path} ({status}, n={problem.n})")

In [ ]:
selected_problems = get_problem_names(PROBLEMS_TO_RUN)
selected_method_options = get_mpc_response_methods(MAX_ITERATIONS)
if METHODS_TO_RUN is not None:
    selected_method_options = [
        entry for entry in selected_method_options if entry[0].label in METHODS_TO_RUN
    ]
    missing = sorted(set(METHODS_TO_RUN) - {entry[0].label for entry in selected_method_options})
    if missing:
        raise ValueError(f"Unknown method labels: {missing}")

tasks = [
    (scenario, seed, problem, method, options)
    for scenario in SCENARIOS_TO_RUN
    for seed in SEEDS
    for problem in selected_problems
    for method, options in selected_method_options
    if METHODS_TO_RUN is not None or method.label in SCENARIO_DEFAULT_LABELS[scenario]
]
print(f"Prepared {len(tasks)} tasks.")
for scenario, seed, problem, method, options in tasks:
    path = result_path(scenario, seed, problem, method.label)
    print(f"{scenario:13s} seed={seed} {problem:24s} {method.label:18s} -> {path}")
    if RUN_EXPERIMENTS:
        run_task(
            scenario, seed, problem, method, options, TIME_LIMIT
        )

if not RUN_EXPERIMENTS:
    print("Dry run only. Set RUN_EXPERIMENTS = True to execute these tasks.")